# 10 — Router (Intent Classifier & Orchestrator)

Tests `agents/router.py`:
- `classify_intents()` JSON output schema
- Correct intent detection for SEARCH, LOGISTICS, PREFERENCE_UPDATE, and combinations
- Allergen & preference extraction
- `route_stream()` end-to-end streaming output
- Sentinel markers (<<PREF_SAVING>>, <<LOGISTICS>>, <<CRITIC>>)

> **Requires**: LLM + Qdrant credentials in `.env`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

In [ ]:
from agents.router import Router

## Fixture: fresh router per test

In [ ]:
def make_router():
    return Router(customer_id='test_customer_nb')

def collect_stream(router, message):
    chunks = []
    for chunk in router.route_stream(message):
        chunks.append(chunk)
        print(chunk, end='', flush=True)
    print()
    return ''.join(chunks)

## 1. classify_intents() schema

In [ ]:
router = make_router()
result = router.classify_intents('I need a birthday cake for my wife who is allergic to nuts.')

import json
print(json.dumps(result, indent=2))

assert isinstance(result, dict), 'Must return a dict'
assert 'intents' in result, 'Must have intents'
assert 'allergies' in result, 'Must have allergies'
assert 'preferences' in result, 'Must have preferences'
assert 'search_query' in result, 'Must have search_query'
assert isinstance(result['intents'], list), 'intents must be a list'
print('\nSchema validation: PASSED')

## 2. SEARCH intent detection

In [ ]:
router = make_router()
messages = [
    'Can you recommend a gift for my mother?',
    'I want to send flowers to my wife.',
    'What cakes do you have under RS.3000?',
]

for msg in messages:
    result = router.classify_intents(msg)
    print(f'{msg!r} → intents: {result["intents"]}')
    assert 'SEARCH' in result['intents'], f'Expected SEARCH in {result["intents"]} for: {msg}'

print('SEARCH detection: PASSED')

## 3. LOGISTICS intent detection

In [ ]:
router = make_router()
messages = [
    'Can you deliver to Kandy?',
    'How long does shipping to Galle take?',
    'Do you deliver to Jaffna by Friday?',
]

for msg in messages:
    result = router.classify_intents(msg)
    print(f'{msg!r} → intents: {result["intents"]}, location: {result.get("location")}')
    assert 'LOGISTICS' in result['intents'], f'Expected LOGISTICS in {result["intents"]} for: {msg}'

print('LOGISTICS detection: PASSED')

## 4. PREFERENCE_UPDATE intent detection

In [ ]:
router = make_router()
messages = [
    'My wife is allergic to nuts.',
    'My mother loves traditional Sri Lankan sweets.',
    'Remember that my dad prefers whisky-flavoured gifts.',
]

for msg in messages:
    result = router.classify_intents(msg)
    print(f'{msg!r} → intents: {result["intents"]}')
    assert 'PREFERENCE_UPDATE' in result['intents'], \
        f'Expected PREFERENCE_UPDATE in {result["intents"]} for: {msg}'

print('PREFERENCE_UPDATE detection: PASSED')

## 5. Multi-intent detection

In [ ]:
router = make_router()
# This message has SEARCH + PREFERENCE_UPDATE + LOGISTICS all at once
msg = 'My wife is allergic to nuts — find me a birthday cake and can you deliver to Colombo by tomorrow?'
result = router.classify_intents(msg)

print('Multi-intent result:', json.dumps(result, indent=2))

intents = result['intents']
assert 'SEARCH' in intents, f'Expected SEARCH in {intents}'
assert 'PREFERENCE_UPDATE' in intents, f'Expected PREFERENCE_UPDATE in {intents}'
assert 'LOGISTICS' in intents, f'Expected LOGISTICS in {intents}'
print('Multi-intent detection: PASSED')

## 6. Allergen extraction

In [ ]:
router = make_router()
result = router.classify_intents(
    'My wife is allergic to nuts and my mother cannot eat gluten.'
)

print('Allergies extracted:', result['allergies'])

assert isinstance(result['allergies'], dict), 'Allergies should be a dict of {recipient: [allergens]}'

# At least one recipient should have allergies extracted
all_allergies = [a for allergen_list in result['allergies'].values() for a in allergen_list]
assert any('nut' in a.lower() for a in all_allergies), \
    f'Nuts allergy not extracted. Got: {result["allergies"]}'

print('Allergen extraction: PASSED')

## 7. Preference extraction

In [ ]:
router = make_router()
result = router.classify_intents(
    'My wife loves vanilla and chocolate flavours.'
)

print('Preferences extracted:', result['preferences'])

assert isinstance(result['preferences'], dict)
all_prefs = [p for pref_list in result['preferences'].values() for p in pref_list]
assert any('vanilla' in p.lower() for p in all_prefs), \
    f'Vanilla preference not extracted. Got: {result["preferences"]}'

print('Preference extraction: PASSED')

## 8. route_stream() — SEARCH only

In [ ]:
router = make_router()
full = collect_stream(router, 'Recommend a flower bouquet for my mom.')

content = full.replace('<<CRITIC>>', '').replace('<<PREF_SAVING>>', '').replace('<<LOGISTICS>>', '').strip()
print(f'\nContent length: {len(content)} chars')
assert len(content) > 50, 'Response should be substantive'
print('route_stream SEARCH: PASSED')

## 9. route_stream() — LOGISTICS only

In [ ]:
router = make_router()
full = collect_stream(router, 'Can you deliver to Kandy?')

content = full.replace('<<LOGISTICS>>', '').strip()
print(f'\nLogistics response: {content[:300]}')
assert len(content) > 20
print('route_stream LOGISTICS: PASSED')

## 10. route_stream() — preference update triggers PREF_SAVING sentinel

In [ ]:
router = make_router()
chunks = []
for chunk in router.route_stream('My wife is allergic to nuts.'):
    chunks.append(chunk)

full = ''.join(chunks)
print('Chunks:', chunks)

assert '<<PREF_SAVING>>' in full, \
    f'Expected <<PREF_SAVING>> sentinel when preference update occurs. Got: {chunks}'
print('PREF_SAVING sentinel: PASSED')